# Composing local operators and using AMG

A local problem exposes its operator, signed trace coupling, source, kernel and physical mean constraints. The global coupling is independent of how these matrices were assembled. CPU AMG acts on a projected and pinned SPD local complement, not directly on the global saddle matrix.

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/dee7b97c4862d02a2de44df2f6306e833f302e5089559cf7aac15c07a0813ad1/09_local_operators_and_amg-companion.zip"
COMPANION_SHA256 = "dee7b97c4862d02a2de44df2f6306e833f302e5089559cf7aac15c07a0813ad1"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("foundations/operators/09_local_operators_and_amg.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace
from examples.formulations.application import brinkman as solve_brinkman
from examples.formulations.application import elasticity as solve_elasticity


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

from pymhm import Equation, LocalEquations, MultiscaleProblem, compile_local_equations

local_forms = LocalEquations(
    [[1.0, -1.0], [-1.0, 1.0]],
    [0.0, 0.0],
    np.eye(2),
    -np.eye(2),
    [0, 1],
    kernel=np.ones((2, 1)),
    moments=np.full((2, 1), 0.5),
)
declared = MultiscaleProblem(Equation(0, [0.0, -1.0, 0.0]), lambda item: local_forms, [0], 2, (1,))
system = assemble(declared)
result = system.solve()
np.testing.assert_allclose(result.fields[0], [0.0, 1.0], atol=1e-12)
local = compile_local_equations(local_forms).problem
lu = local.condense("scipy")
amg = local.condense("pyamg")
np.testing.assert_allclose(lu.lifts, amg.lifts, atol=1e-12)
mesh = TriangleMesh.unit_square(2)
solution_definition = define_darcy(mesh, dirichlet=lambda x: 1 + x[:, 0])
solution_system = assemble(solution_definition.problem, solvers=SolverConfig(local_solver="pyamg"))
solution = recover_darcy(
    solution_definition,
    solution_system,
    solution_system.solve(constraints=pressure_constraints(solution_definition, solution_system)),
)
assert solution.l2_error(lambda x: 1 + x[:, 0]) < 1e-09
print("AMG MHM pressure error", solution.l2_error(lambda x: 1 + x[:, 0]))


GPU AMG uses the optional AmgX adapter. CPU/GPU backends must meet the same residual criterion, and transfer/setup costs belong in timing. No novelty claim about the first use of AMG with MHM is made.